In [ ]:
"""评估城市层级与3CE存量/增量市场之间的关系。

将本脚本和“3CE问卷数据-2026-08-12.csv”放在同一文件夹，直接运行。
依赖：pip install pandas openpyxl matplotlib

口径：
1. 存量市场 = 经常购买 + 买过1-2次（已经产生过购买）。
2. 增量市场 = 听说过但没有购买 + 完全不了解（尚未购买）。
3. 问卷样本不是全国人口加权样本，结果反映样本内部关系，不能直接当作全国市场规模。
"""

In [ ]:
from __future__ import annotations

In [ ]:
import argparse
import math
import re
from pathlib import Path

In [ ]:
import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

In [ ]:
CITY_COLUMN = "所在城市级别"
Q9_COLUMN = "3CE了解程度"
ID_COLUMN = "记录ID"
CITY_ORDER = ["一线城市", "新一线城市", "二线城市", "三线及以下城市/县城"]
MARKET_ORDER = ["存量市场", "增量市场"]
STAGE_ORDER = [
    "核心存量｜经常购买",
    "待激活存量｜买过1-2次",
    "潜在增量｜听说过未购买",
    "全新增量｜完全不了解",
]

In [ ]:
def read_csv(path: Path) -> pd.DataFrame:
    errors = []
    for encoding in ("utf-8-sig", "utf-8", "gb18030"):
        try:
            return pd.read_csv(path, encoding=encoding, dtype=str, keep_default_na=False)
        except (UnicodeDecodeError, pd.errors.ParserError) as exc:
            errors.append(f"{encoding}: {exc}")
    raise RuntimeError("无法读取CSV。\n" + "\n".join(errors))

In [ ]:
def clean(value: object) -> str:
    return re.sub(r"\s+", " ", str(value)).strip() if value is not None else ""

In [ ]:
def normalize_city(value: object) -> str:
    text = clean(value).replace(" ", "")
    if "新一线" in text:
        return "新一线城市"
    if text.startswith("一线"):
        return "一线城市"
    if "二线" in text:
        return "二线城市"
    if any(x in text for x in ["三线", "四线", "五线", "六线", "县城", "乡镇", "农村", "低线"]):
        return "三线及以下城市/县城"
    return "城市未识别"

In [ ]:
def normalize_stage(value: object) -> str:
    text = clean(value).replace(" ", "")
    if "经常购买" in text:
        return "核心存量｜经常购买"
    if "买过" in text or re.search(r"1[-—至~]2", text):
        return "待激活存量｜买过1-2次"
    if "听说过" in text or "未购买" in text or "没有购买" in text:
        return "潜在增量｜听说过未购买"
    if "完全不了解" in text:
        return "全新增量｜完全不了解"
    return "市场阶段未识别"

In [ ]:
def stage_to_market(stage: str) -> str:
    if stage.startswith("核心存量") or stage.startswith("待激活存量"):
        return "存量市场"
    if stage.startswith("潜在增量") or stage.startswith("全新增量"):
        return "增量市场"
    return "市场类型未识别"

In [ ]:
def gamma_q(a: float, x: float) -> float:
    """正则化上不完全伽马函数Q(a,x)，用于纯Python计算卡方检验p值。"""
    if a <= 0 or x < 0:
        return float("nan")
    if x == 0:
        return 1.0
    eps = 3e-14
    max_iter = 1000
    fp_min = 1e-300
    log_term = -x + a * math.log(x) - math.lgamma(a)
    if x < a + 1.0:
        ap = a
        total = 1.0 / a
        delta = total
        for _ in range(max_iter):
            ap += 1.0
            delta *= x / ap
            total += delta
            if abs(delta) < abs(total) * eps:
                break
        lower = total * math.exp(log_term)
        return max(0.0, min(1.0, 1.0 - lower))

    b = x + 1.0 - a
    c = 1.0 / fp_min
    d = 1.0 / max(abs(b), fp_min)
    if b < 0:
        d = -d
    h = d
    for i in range(1, max_iter + 1):
        an = -i * (i - a)
        b += 2.0
        d = an * d + b
        if abs(d) < fp_min:
            d = fp_min
        c = b + an / c
        if abs(c) < fp_min:
            c = fp_min
        d = 1.0 / d
        delta = d * c
        h *= delta
        if abs(delta - 1.0) < eps:
            break
    return max(0.0, min(1.0, math.exp(log_term) * h))

In [ ]:
def chi_square_test(observed: pd.DataFrame) -> tuple[float, int, float, pd.DataFrame]:
    total = observed.to_numpy().sum()
    row_totals = observed.sum(axis=1)
    col_totals = observed.sum(axis=0)
    expected = pd.DataFrame(index=observed.index, columns=observed.columns, dtype=float)
    for row in observed.index:
        for col in observed.columns:
            expected.loc[row, col] = row_totals[row] * col_totals[col] / total
    chi2 = float((((observed - expected) ** 2) / expected).to_numpy().sum())
    degrees = (observed.shape[0] - 1) * (observed.shape[1] - 1)
    p_value = gamma_q(degrees / 2.0, chi2 / 2.0)
    return chi2, degrees, p_value, expected

In [ ]:
def cramers_v(chi2: float, n: int, rows: int, cols: int) -> float:
    denominator = n * min(rows - 1, cols - 1)
    return math.sqrt(chi2 / denominator) if denominator > 0 else float("nan")

In [ ]:
def v_interpretation(value: float) -> str:
    if pd.isna(value):
        return "无法判断"
    if value < 0.10:
        return "几乎无关联"
    if value < 0.30:
        return "弱关联"
    if value < 0.50:
        return "中等关联"
    return "强关联"

In [ ]:
def wilson_interval(successes: int, total: int, z: float = 1.96) -> tuple[float, float]:
    if total == 0:
        return float("nan"), float("nan")
    p = successes / total
    denominator = 1 + z * z / total
    center = (p + z * z / (2 * total)) / denominator
    margin = z * math.sqrt((p * (1 - p) + z * z / (4 * total)) / total) / denominator
    return max(0.0, center - margin), min(1.0, center + margin)

In [ ]:
def adjusted_residuals(observed: pd.DataFrame, expected: pd.DataFrame) -> pd.DataFrame:
    """调整标准化残差：绝对值≥1.96代表该格与独立假设有明显偏离。"""
    total = observed.to_numpy().sum()
    row_prop = observed.sum(axis=1) / total
    col_prop = observed.sum(axis=0) / total
    result = pd.DataFrame(index=observed.index, columns=observed.columns, dtype=float)
    for row in observed.index:
        for col in observed.columns:
            denominator = math.sqrt(expected.loc[row, col] * (1 - row_prop[row]) * (1 - col_prop[col]))
            result.loc[row, col] = (observed.loc[row, col] - expected.loc[row, col]) / denominator
    return result

In [ ]:
def city_metrics(df: pd.DataFrame, observed: pd.DataFrame) -> pd.DataFrame:
    overall_stock_rate = df["市场类型"].eq("存量市场").mean()
    overall_increment_rate = df["市场类型"].eq("增量市场").mean()
    rows = []
    for city in observed.index:
        stock = int(observed.loc[city, "存量市场"])
        increment = int(observed.loc[city, "增量市场"])
        total = stock + increment
        stock_rate = stock / total
        increment_rate = increment / total
        stock_low, stock_high = wilson_interval(stock, total)
        inc_low, inc_high = wilson_interval(increment, total)
        rows.append([
            city, total, total / len(df), stock, stock_rate, stock_low, stock_high,
            stock_rate / overall_stock_rate * 100 if overall_stock_rate else float("nan"),
            increment, increment_rate, inc_low, inc_high,
            increment_rate / overall_increment_rate * 100 if overall_increment_rate else float("nan"),
            stock / len(df), increment / len(df), increment_rate - overall_increment_rate,
        ])
    columns = [
        "城市层级", "样本量", "样本占比", "存量人数", "存量率", "存量率95%CI下限", "存量率95%CI上限",
        "存量指数（总体=100）", "增量人数", "增量率", "增量率95%CI下限", "增量率95%CI上限",
        "增量指数（总体=100）", "存量机会贡献", "增量机会贡献", "增量率较总体差值",
    ]
    return pd.DataFrame(rows, columns=columns)

In [ ]:
def odds_ratios(observed: pd.DataFrame, reference: str = "一线城市") -> pd.DataFrame:
    """比较各城市成为存量用户的优势比；一线城市为参照。"""
    if reference not in observed.index:
        reference = observed.index[0]
    ref_stock = float(observed.loc[reference, "存量市场"])
    ref_increment = float(observed.loc[reference, "增量市场"])
    rows = []
    for city in observed.index:
        stock = float(observed.loc[city, "存量市场"])
        increment = float(observed.loc[city, "增量市场"])
        # Haldane-Anscombe修正避免零频数。
        a, b, c, d = stock + 0.5, increment + 0.5, ref_stock + 0.5, ref_increment + 0.5
        odds_ratio = (a / b) / (c / d)
        standard_error = math.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
        low = math.exp(math.log(odds_ratio) - 1.96 * standard_error)
        high = math.exp(math.log(odds_ratio) + 1.96 * standard_error)
        rows.append([
            city, reference, int(stock), int(increment), odds_ratio, low, high,
            "存量倾向显著更高" if low > 1 else ("存量倾向显著更低" if high < 1 else "差异不显著"),
        ])
    return pd.DataFrame(rows, columns=[
        "城市层级", "参照城市", "存量人数", "增量人数", "存量优势比OR",
        "OR 95%CI下限", "OR 95%CI上限", "判断",
    ])

In [ ]:
def stage_detail(df: pd.DataFrame) -> pd.DataFrame:
    count = pd.crosstab(df["城市层级"], df["市场阶段"]).reindex(
        index=[x for x in CITY_ORDER if x in df["城市层级"].unique()],
        columns=STAGE_ORDER,
        fill_value=0,
    )
    rate = count.div(count.sum(axis=1), axis=0)
    rows = []
    for city in count.index:
        for stage in count.columns:
            rows.append([city, stage, int(count.loc[city, stage]), float(rate.loc[city, stage])])
    return pd.DataFrame(rows, columns=["城市层级", "市场阶段", "人数", "城市内占比"])

In [ ]:
def format_workbook(writer: pd.ExcelWriter) -> None:
    header_fill = PatternFill("solid", fgColor="8F315B")
    for ws in writer.book.worksheets:
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        for cell in ws[1]:
            cell.font = Font(name="微软雅黑", size=11, bold=True, color="FFFFFF")
            cell.fill = header_fill
            cell.alignment = Alignment(horizontal="center", vertical="center")
        for cells in ws.columns:
            values = list(cells)
            width = min(max(len(str(cell.value or "")) for cell in values[:300]) + 2, 42)
            ws.column_dimensions[values[0].column_letter].width = max(width, 11)
        for row in ws.iter_rows():
            for cell in row:
                header = clean(ws.cell(1, cell.column).value)
                if isinstance(cell.value, float) and any(x in header for x in ["率", "比例", "占比", "贡献", "差值"]):
                    cell.number_format = "0.0%"
                elif isinstance(cell.value, float):
                    cell.number_format = "0.000"

In [ ]:
def create_charts(metrics: pd.DataFrame, output_dir: Path) -> None:
    if plt is None:
        print("提示：未安装matplotlib，已跳过PNG图表；Excel分析不受影响。")
        return
    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS"]
    plt.rcParams["axes.unicode_minus"] = False

    plot = metrics.set_index("城市层级")[["存量率", "增量率"]]
    ax = plot.plot.bar(stacked=True, figsize=(10, 6), color=["#8F315B", "#E8B4C8"])
    ax.set_ylim(0, 1); ax.set_xlabel(""); ax.set_ylabel("城市内占比")
    ax.set_title("不同城市层级的3CE存量与增量结构")
    ax.legend(loc="upper right")
    for container in ax.containers:
        ax.bar_label(container, labels=[f"{x:.0%}" if x >= 0.08 else "" for x in container.datavalues], label_type="center")
    plt.xticks(rotation=0); plt.tight_layout()
    plt.savefig(output_dir / "城市层级_存量增量结构.png", dpi=180); plt.close()

    plot = metrics.set_index("城市层级")[["存量指数（总体=100）", "增量指数（总体=100）"]]
    ax = plot.plot.bar(figsize=(11, 6), color=["#7A274C", "#D978A3"])
    ax.axhline(100, color="#555555", linestyle="--", linewidth=1)
    ax.set_xlabel(""); ax.set_ylabel("指数（总体=100）")
    ax.set_title("不同城市层级的存量/增量指数")
    plt.xticks(rotation=0); plt.tight_layout()
    plt.savefig(output_dir / "城市层级_存量增量指数.png", dpi=180); plt.close()

In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser(description="城市层级与3CE存量/增量市场关系评估")
    script_dir = Path(__file__).resolve().parent
    parser.add_argument(
        "csv", type=Path, nargs="?",
        default=script_dir / "3CE问卷数据-2026-08-12.csv",
        help="问卷CSV；不填写时读取脚本同目录的默认文件",
    )
    parser.add_argument(
        "-o", "--output", type=Path,
        default=script_dir / "城市层级与存量增量分析结果",
        help="结果文件夹",
    )
    args = parser.parse_args()
    if not args.csv.exists():
        raise FileNotFoundError(
            f"找不到CSV：{args.csv}\n请把脚本与3CE问卷数据-2026-08-12.csv放在同一文件夹。"
        )
    args.output.mkdir(parents=True, exist_ok=True)

    raw = read_csv(args.csv)
    missing = [column for column in [CITY_COLUMN, Q9_COLUMN] if column not in raw.columns]
    if missing:
        raise KeyError("CSV缺少字段：" + "、".join(missing))

    df = raw.copy()
    df["城市层级"] = df[CITY_COLUMN].map(normalize_city)
    df["市场阶段"] = df[Q9_COLUMN].map(normalize_stage)
    df["市场类型"] = df["市场阶段"].map(stage_to_market)
    valid = df[
        df["城市层级"].isin(CITY_ORDER) & df["市场类型"].isin(MARKET_ORDER)
    ].copy()
    if valid.empty:
        raise ValueError("没有可用于城市层级×存量增量分析的有效数据。")

    observed = pd.crosstab(valid["城市层级"], valid["市场类型"]).reindex(
        index=[x for x in CITY_ORDER if x in valid["城市层级"].unique()],
        columns=MARKET_ORDER,
        fill_value=0,
    )
    chi2, degrees, p_value, expected = chi_square_test(observed)
    association = cramers_v(chi2, len(valid), observed.shape[0], observed.shape[1])
    residuals = adjusted_residuals(observed, expected)
    metrics = city_metrics(valid, observed)
    odds = odds_ratios(observed)
    stages = stage_detail(valid)

    significance = "显著相关" if p_value < 0.05 else "没有发现显著相关"
    test_summary = pd.DataFrame([
        ["有效样本量", len(valid), "用于城市×存量/增量分析的记录"],
        ["卡方统计量", chi2, "越大代表观察结构偏离独立假设越明显"],
        ["自由度", degrees, "(城市类别数-1)×(市场类型数-1)"],
        ["p值", p_value, "p<0.05通常判断为统计显著"],
        ["显著性结论", significance, "统计相关不等同于城市层级造成购买"],
        ["Cramér's V", association, "0-1之间；越大代表关联越强"],
        ["关联强度", v_interpretation(association), "<0.1几乎无；0.1-0.3弱；0.3-0.5中等；≥0.5强"],
        ["最小期望频数", expected.to_numpy().min(), "通常应≥5；过低时卡方近似需谨慎"],
    ], columns=["指标", "结果", "解释"])

    count_table = observed.reset_index()
    count_table.columns.name = None
    row_rate = observed.div(observed.sum(axis=1), axis=0).reset_index()
    row_rate.columns.name = None
    expected_table = expected.reset_index(); expected_table.columns.name = None
    residual_table = residuals.reset_index(); residual_table.columns.name = None

    residual_long = residuals.stack().rename("调整标准化残差").reset_index()
    residual_long.columns = ["城市层级", "市场类型", "调整标准化残差"]
    residual_long["判断"] = residual_long["调整标准化残差"].map(
        lambda x: "显著高于期望" if x >= 1.96 else ("显著低于期望" if x <= -1.96 else "与期望无明显差异")
    )

    definitions = pd.DataFrame([
        ["存量市场", "Q9=经常购买或买过1-2次", "代表已经产生过3CE购买"],
        ["增量市场", "Q9=听说过但没有购买或完全不了解", "代表尚未产生3CE购买"],
        ["存量/增量指数", "城市对应比例÷总体对应比例×100", "100=总体平均；>100表示该城市相对集中"],
        ["机会贡献", "城市对应人数÷全部有效样本", "样本内部的机会构成，不是全国市场份额"],
        ["调整标准化残差", "观察人数与独立假设期望人数的标准化差异", "≥1.96显著偏高；≤-1.96显著偏低"],
        ["优势比OR", "城市存量优势÷一线城市存量优势", ">1更倾向存量；95%CI跨1代表差异不显著"],
    ], columns=["项目", "计算口径", "解读"])

    output_xlsx = args.output / "城市层级与3CE存量增量关系评估.xlsx"
    with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
        test_summary.to_excel(writer, sheet_name="统计结论", index=False)
        metrics.to_excel(writer, sheet_name="城市核心指标", index=False)
        count_table.to_excel(writer, sheet_name="人数交叉表", index=False)
        row_rate.to_excel(writer, sheet_name="城市内结构占比", index=False)
        expected_table.to_excel(writer, sheet_name="独立假设期望人数", index=False)
        residual_table.to_excel(writer, sheet_name="标准化残差矩阵", index=False)
        residual_long.to_excel(writer, sheet_name="残差判断", index=False)
        odds.to_excel(writer, sheet_name="相对一线优势比", index=False)
        stages.to_excel(writer, sheet_name="四阶段城市明细", index=False)
        definitions.to_excel(writer, sheet_name="口径说明", index=False)
        detail_columns = [x for x in [ID_COLUMN, CITY_COLUMN, Q9_COLUMN, "城市层级", "市场阶段", "市场类型"] if x in valid.columns]
        valid[detail_columns].to_excel(writer, sheet_name="有效样本明细", index=False)
        format_workbook(writer)

    create_charts(metrics, args.output)

    print("\n========== 城市层级与存量/增量关系 ==========")
    print(f"有效样本：{len(valid)}人")
    print(f"卡方检验：统计量={chi2:.3f}，自由度={degrees}，p={p_value:.4f}，结论={significance}")
    print(f"关联强度：Cramers V={association:.3f}，判断={v_interpretation(association)}")
    print("\n【各城市核心指标】")
    print(metrics[[
        "城市层级", "样本量", "存量率", "增量率", "存量指数（总体=100）",
        "增量指数（总体=100）", "增量机会贡献",
    ]].round(3).to_string(index=False))
    print("\n【显著偏离独立假设的单元格】")
    important = residual_long[residual_long["调整标准化残差"].abs().ge(1.96)]
    print(important.round(3).to_string(index=False) if not important.empty else "没有绝对残差≥1.96的单元格")
    print(f"\nExcel：{output_xlsx.resolve()}")
    print("=============================================")

In [ ]:
if __name__ == "__main__":
    main()